# 실습 D. 재검증 범위 줄이기 (약 20분)

> **코드가 바뀌어도 Dafny 증명을 전부 다시 할 필요는 없다. 무엇이 바뀌었느냐에 따라 다시 증명할 범위가 정해진다. 그 범위는 Java 영향 분석의 범위와 다르다.**

Dafny는 메서드를 하나씩 증명하고, 다른 메서드를 부를 때는 **그 메서드의 계약만** 봅니다. 그래서 다음 규칙이 나옵니다.

| 바뀐 것 | 다시 증명할 것 |
|---|---|
| 메서드 본문만 | 그 메서드 하나 (부르는 쪽은 계약만 봤으므로) |
| 메서드 계약 | 그 메서드 + 부르는 메서드 |

Java가 바뀌면 Dafny 모델도 새 코드대로 고쳐 둡니다(여기서는 `scenarios/req*/model/`에 고쳐 두었습니다).

In [ ]:
import lab   # 실습 도우미: lab.run(...)이 make 명령을 실행하고 결과 보고서를 아래에 보여 줍니다

## 변경 요청 2: 본문만 바뀐 경우

**요청 (플랫폼팀).** "허브 칩에서 나눗셈(`%`)이 느리니, 대기열 `RingEventQueue`에서 `%`를 없애 달라. `clear()`는 `poll()`을 반복하지 말고 한 번에 비워 달라. 동작은 바꾸지 말 것."

AI는 Java 메서드 **네 개의 본문**을 고쳤습니다. 셋은 `%`를 쓰던 곳이고, 하나는 비우는 방식을 바꾼 곳입니다.

| 메서드 | 레거시 | 변경 요청 2 |
|---|---|---|
| `offer` | `buf[(head + count) % buf.length] = e` | `idx = head + count`, 끝을 넘으면 `idx -= buf.length` |
| `poll` | `head = (head + 1) % buf.length` | `head++`, 끝에 닿으면 `head = 0` |
| `snapshot` | `buf[(head + i) % buf.length]`를 차례로 담음 | `idx`를 하나씩 늘리다 끝에 닿으면 0 |
| `clear` | 빌 때까지 `poll()` 반복 | `Arrays.fill(buf, null); head = 0; count = 0` |

Dafny 모델도 **이 네 메서드(`Offer`, `Poll`, `Snapshot`, `Clear`)의 본문만** 새 Java 코드대로 고쳤습니다. 계약(`requires`/`ensures`)은 한 줄도 바뀌지 않았습니다. 아래 diff에서 바뀐 곳마다 위에 메서드 이름(▸)이 붙어 있습니다.

왼쪽 `formal/RingQueue.dfy`는 레거시 대기열을 옮긴 Dafny 모델, 오른쪽은 이번 변경대로 고친 모델입니다. `Wrap(k)`는 이 Dafny 모델 안에서 정의한 함수로, Java 코드에 직접 쓰인 계산 `k % buf.length`를 나타냅니다(Java API가 아님). `i32`는 Java `int`(32비트) 범위의 정수, `x as int`는 그 값을 Dafny의 정수로 쓴다는 뜻입니다.


In [1]:
lab.diff("formal/RingQueue.dfy", "scenarios/req2/model/RingQueue.dfy", title="대기열 모델: 레거시(formal/) → 변경 요청 2")

In [2]:
lab.run("reverify", REQ="req2")

$ make reverify REQ=req2


49개 중 **4개**만 다시 증명합니다. 본문이 바뀐 `Offer`, `Poll`, `Snapshot`, `Clear` 자신입니다. 계약이 그대로이므로, 이 메서드를 부르는 쪽은 다시 증명할 필요가 없습니다.

같은 변경에 Java 영향 분석 도구(`lab.run("impact", V="scenarios/req2/ai")`, 바뀐 값이 흐르는 곳을 따라가는 도구)를 돌리면 메서드 79개가 "달라질 수 있다"고 나옵니다. Dispatcher나 전력 메서드의 Dafny 증명은 대기열 **코드**가 아니라 대기열의 **계약**만 보고 이뤄졌고, 계약은 그대로이므로 다시 증명할 필요가 없습니다.

## 변경 요청 1: 계약이 바뀐 경우

**요청 (상품기획팀).** "평소에는 대기열 8칸만 쓰고, 이벤트가 몰리면 최대 32칸까지 늘려 달라." AI 패치 B는 대기열이 가득 차면 배열을 늘리고, 상한 32는 Dispatcher가 설정값으로 직접 갖게 했습니다. 그래서 대기열의 **계약**이 바뀌었습니다(가득 차서 FULL이 나오는 때가 달라지고, 배열이 새것으로 바뀔 수 있음).

**이 패치를 내보내려면 무엇을 다시 증명해야 할까요?**

In [3]:
lab.run("reverify", REQ="req1")

$ make reverify REQ=req1


51개 중 **19개**. 대기열 계약이 바뀌었으므로 대기열 대부분과, 그 계약을 믿고 증명된 Dispatcher까지 다시 증명합니다. 반대로 전력 모델은 대기열을 언급하지 않으므로 다시 증명하지 않습니다(Java 영향 분석은 대기열에서 꺼낸 이벤트가 전력 승인까지 흘러가므로 전력 메서드도 "달라질 수 있다"고 봅니다).

**생각해 볼 것.** Java 영향 분석의 범위와 Dafny 재검증 범위가 다른 이유를 한 문장으로 설명해 보세요.